# 02. 데이터 병합 — 음성 + 썸네일 피처 조인

video_cleaned(기본 전처리 완료) 데이터에 채널별 음성 피처와 영상별 썸네일 피처를 left join으로 병합한다.

| 단계 | 소스 | 키 | 병합 컬럼 |
|------|------|-----|----------|
| 1. 음성 | `channels_voice.csv` | `channel_id` | wpm, f0_mean, f0_std, hnr_mean, speech_ratio |
| 2. 썸네일 | `thumbnail_all.csv` | `thumbnail_url` | face_area_ratio, face_count, avg_brightness, avg_saturation, contrast, text_area_ratio, color_variance |

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

BASE_PATH = Path("../../data")
FEAT_PATH = BASE_PATH / "features"

## 1. 데이터 로드

In [ ]:
# 데이터 로드: gemini 병합본 + 음성/썸네일/버츄얼 원천
df = pd.read_csv(BASE_PATH / "processed/video_merged_gemini.csv", sep="\x01", low_memory=False)
df_voice = pd.read_csv(FEAT_PATH / "voice/channels_voice.csv", encoding="cp949")
df_thumb = pd.read_csv(FEAT_PATH / "thumbnail/thumbnail_all.csv", sep="\x01", low_memory=False)
df_vtuber = pd.read_excel(FEAT_PATH / "image/vtuber_scores.xlsx")

print(f"df       : {df.shape}")
print(f"df_voice : {df_voice.shape}")
print(f"df_thumb : {df_thumb.shape}")
print(f"df_vtuber: {df_vtuber.shape}")

In [ ]:
# Gemini 피처 결측 행 제거 (모델 피처라 없으면 사용 불가)
before = len(df)
df = df[df['real_camera_real_cam_level'].notna()].reset_index(drop=True)
print(f"Gemini 결측 행 제거: {before} -> {len(df)} (제거: {before - len(df)}건)")

---
## 2. 음성 피처 병합 (channel_id)

| 피처 | 설명 |
|------|------|
| wpm | 분당 단어 수 (말의 빠르기) |
| f0_mean | 평균 음높이 |
| f0_std | 음높이 표준편차 |
| hnr_mean | 하모니 대비 노이즈 비율 (클수록 맑음) |
| speech_ratio | 전체 중 말한 시간 비율 |

In [3]:
# 병합 키 검증
v_ids = set(df["channel_id"].unique())
voice_ids = set(df_voice["channel_id"].dropna().unique())

print(f"df 채널 수       : {len(v_ids)}")
print(f"df_voice 채널 수 : {len(voice_ids)}")
print(f"공통              : {len(v_ids & voice_ids)}")
print(f"df에만 있음       : {len(v_ids - voice_ids)}  (병합 후 NaN)")
print(f"df_voice에만 있음 : {len(voice_ids - v_ids)}")

df 채널 수       : 99
df_voice 채널 수 : 211
공통              : 99
df에만 있음       : 0  (병합 후 NaN)
df_voice에만 있음 : 112


In [ ]:
VOICE_COLS = ['wpm', 'f0_mean', 'f0_std', 'hnr_mean', 'speech_ratio']

# 병합할 음성 컬럼만 추출 + channel_id 결측 제거
voice_merge = df_voice[['channel_id'] + VOICE_COLS].dropna(subset=['channel_id'])
print(f"voice dropna 전: {len(df_voice)} -> 후: {len(voice_merge)}")

# channel_id 기준 left join
before = df.shape
df = df.merge(voice_merge, on='channel_id', how='left')
print(f"병합: {before} -> {df.shape}")

# NaN 검증
print("\n[음성 피처 NaN 비율]")
print((df[VOICE_COLS].isna().mean() * 100).round(2).to_string())

---
## 3. 썸네일 피처 병합 (thumbnail_url)

| 피처 | 설명 |
|------|------|
| face_area_ratio | 얼굴 면적 비율 |
| face_count | 얼굴 개수 |
| avg_brightness | 평균 밝기 |
| avg_saturation | 평균 채도 |
| contrast | 명암 대비 |
| text_area_ratio | 텍스트 면적 비율 |
| color_variance | 색 분산도 |

In [5]:
THUMB_COLS = ['face_area_ratio', 'face_count', 'avg_brightness',
              'avg_saturation', 'contrast', 'text_area_ratio', 'color_variance']

# 컬럼 충돌 확인
overlap = set(df.columns) & set(THUMB_COLS)
if overlap:
    print(f"WARNING: 기존 df에 이미 존재하는 컬럼: {overlap}")
else:
    print("컬럼 충돌 없음")

컬럼 충돌 없음


In [6]:
# 병합 키 검증
v_urls = set(df['thumbnail_url'].dropna().unique())
t_urls = set(df_thumb['thumbnail_url'].dropna().unique())

print(f"df URL 수        : {len(v_urls)}")
print(f"df_thumb URL 수  : {len(t_urls)}")
print(f"공통              : {len(v_urls & t_urls)}")
print(f"df에만 있음       : {len(v_urls - t_urls)}  (병합 후 NaN)")
print(f"df_thumb에만 있음 : {len(t_urls - v_urls)}")

df URL 수        : 14824
df_thumb URL 수  : 76465
공통              : 14823
df에만 있음       : 1  (병합 후 NaN)
df_thumb에만 있음 : 61642


In [7]:
# 썸네일 데이터 중복 제거
# text_area_ratio 값이 있는 행 우선 (na_position='last'), thumbnail_url 기준 dedup
dup_count = df_thumb['thumbnail_url'].duplicated().sum()
df_thumb_dedup = (
    df_thumb
    .sort_values('text_area_ratio', na_position='last')
    .drop_duplicates(subset=['thumbnail_url'], keep='first')
    .reset_index(drop=True)
)
assert df_thumb_dedup['thumbnail_url'].duplicated().sum() == 0, "thumbnail_url 중복 제거 실패"
print(f"썸네일 중복 제거: {len(df_thumb)} -> {len(df_thumb_dedup)} (제거: {dup_count}건)")

# Left Join
before = df.shape
df = df.merge(
    df_thumb_dedup[['thumbnail_url'] + THUMB_COLS],
    on='thumbnail_url', how='left'
)
print(f"병합: {before} -> {df.shape}")

# NaN 검증
thumb_na_count = df['color_variance'].isna().sum()
print(f"\n[썸네일 피처 결측 행: {thumb_na_count}건 / {len(df)}건 ({thumb_na_count/len(df)*100:.1f}%)]")
print((df[THUMB_COLS].isna().mean() * 100).round(2).to_string())

썸네일 중복 제거: 76465 -> 76465 (제거: 0건)
병합: (14824, 83) -> (14824, 90)

[썸네일 피처 결측 행: 1건 / 14824건 (0.0%)]
face_area_ratio    0.01
face_count         0.01
avg_brightness     0.01
avg_saturation     0.01
contrast           0.01
text_area_ratio    0.01
color_variance     0.01


In [8]:
# 썸네일 피처 결측 행 드랍 (기존 코드와 동일)
missing_mask = df['color_variance'].isna()
missing_count = missing_mask.sum()
print(f"썸네일 피처 결측: {missing_count}건 -> 드랍")

df = df[~missing_mask].reset_index(drop=True)
print(f"드랍 후 shape: {df.shape}")

썸네일 피처 결측: 1건 -> 드랍
드랍 후 shape: (14823, 90)


In [9]:
# text_area_ratio 결측치 -> 0으로 채우기
tar_na = df['text_area_ratio'].isna().sum()
print(f"text_area_ratio 결측: {tar_na}건")
if tar_na > 0:
    df['text_area_ratio'] = df['text_area_ratio'].fillna(0)
    print(f"fillna(0) 후 결측: {df['text_area_ratio'].isna().sum()}건")

text_area_ratio 결측: 0건


---
## 3-1. 버츄얼 이미지 점수 병합 (channel_title → person_folder)

`vtuber_scores.xlsx`는 채널별 이미지 4장의 외형 평가 점수 데이터.
정규화 매칭(`normalize_name`) + 채널 단위 mean 집계 후 병합한다.

| 피처 | 설명 |
|------|------|
| vtuber_features_score | 디자인 특징 점수 |
| vtuber_vibe_score | 분위기/바이브 점수 |
| vtuber_styling_score | 스타일링 점수 |
| vtuber_technical_score | 기술적 완성도 점수 |
| vtuber_rendering_score | 렌더링 품질 점수 |
| vtuber_finish_score | 마감 품질 점수 |
| vtuber_beauty_score | 미적 점수 |
| vtuber_qual_score | 종합 품질 점수 |
| vtuber_appeal_score | 어필 점수 |

In [10]:
import re

def normalize_name(name):
    """영문자, 숫자, 한글, 일본어만 남기고 모든 특수문자와 공백 제거"""
    return re.sub(r'[^a-zA-Z0-9가-힣ㄱ-ㅎㅏ-ㅣ\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', '', str(name))

VTUBER_SCORE_COLS = ['features_score', 'vibe_score', 'styling_score', 'technical_score',
                     'rendering_score', 'finish_score', 'beauty_score', 'qual_score',
                     'vtuber_appeal_score']

# 채널별 mean 집계 (이미지 4장 → 1행)
vtuber_agg = (
    df_vtuber
    .groupby('person_folder')[VTUBER_SCORE_COLS]
    .mean()
    .reset_index()
)
print(f"채널별 집계: {len(df_vtuber)} -> {len(vtuber_agg)}")

# 수동 매핑: 정규화로도 매칭 안 되는 채널 (채널명 변경 / 오타)
MANUAL_MAP = {
    '순망': '겜순망',
    '인간아이쨩입니다': '인간아이짱입니다',
    '미녕이데려오깨': '미녕이데려오께',
}
vtuber_agg['person_folder'] = vtuber_agg['person_folder'].replace(
    {v: k for k, v in MANUAL_MAP.items()}
)
print(f"수동 매핑 적용: {len(MANUAL_MAP)}건")

# 정규화 키 생성
df['_norm'] = df['channel_title'].apply(normalize_name)
vtuber_agg['_norm'] = vtuber_agg['person_folder'].apply(normalize_name)

# 병합 키 검증
v_norms = set(df['_norm'].unique())
vt_norms = set(vtuber_agg['_norm'].unique())
print(f"\ndf 채널 수         : {len(v_norms)}")
print(f"df_vtuber 채널 수  : {len(vt_norms)}")
print(f"공통                : {len(v_norms & vt_norms)}")
print(f"df에만 있음         : {len(v_norms - vt_norms)}  (병합 후 NaN)")

# vtuber_ 접두어 추가 (vtuber_appeal_score는 이미 있으므로 제외)
rename_map = {c: f'vtuber_{c}' for c in VTUBER_SCORE_COLS if not c.startswith('vtuber_')}
vtuber_agg = vtuber_agg.rename(columns=rename_map)
VTUBER_COLS = [rename_map.get(c, c) for c in VTUBER_SCORE_COLS]

# 컬럼 충돌 확인
col_overlap = set(df.columns) & set(VTUBER_COLS)
if col_overlap:
    print(f"WARNING: 기존 df에 이미 존재하는 컬럼: {col_overlap}")
else:
    print("컬럼 충돌 없음")

# Left Join
before = df.shape
df = df.merge(vtuber_agg[['_norm'] + VTUBER_COLS], on='_norm', how='left')
df = df.drop(columns=['_norm'])
print(f"\n병합: {before} -> {df.shape}")

# NaN 검증
print("\n[버츄얼 이미지 피처 NaN 비율]")
print((df[VTUBER_COLS].isna().mean() * 100).round(2).to_string())

df[VTUBER_COLS].head(1)

채널별 집계: 1056 -> 264
수동 매핑 적용: 3건

df 채널 수         : 99
df_vtuber 채널 수  : 264
공통                : 99
df에만 있음         : 0  (병합 후 NaN)
컬럼 충돌 없음



병합: (14823, 91) -> (14823, 99)

[버츄얼 이미지 피처 NaN 비율]
vtuber_features_score     0.0
vtuber_vibe_score         0.0
vtuber_styling_score      0.0
vtuber_technical_score    0.0
vtuber_rendering_score    0.0
vtuber_finish_score       0.0
vtuber_beauty_score       0.0
vtuber_qual_score         0.0
vtuber_appeal_score       0.0


,vtuber_features_score,vtuber_vibe_score,vtuber_styling_score,vtuber_technical_score,vtuber_rendering_score,vtuber_finish_score,vtuber_beauty_score,vtuber_qual_score,vtuber_appeal_score
0,59.066117,55.129598,58.412394,61.87138,65.550305,47.679654,59.412635,64.241419,59.412635


---
## 4. 전체 결과 검증

In [11]:
ALL_NEW_COLS = VOICE_COLS + THUMB_COLS + VTUBER_COLS

print(f"최종 shape: {df.shape}")
print(f"추가된 컬럼 ({len(ALL_NEW_COLS)}개): {ALL_NEW_COLS}")

# 결측 현황 요약
na_summary = df[ALL_NEW_COLS].isna().sum()
na_summary = na_summary[na_summary > 0]

if na_summary.empty:
    print("\n추가 피처 결측치 없음")
else:
    print(f"\n[결측 있는 피처: {len(na_summary)}개]")
    print(na_summary.to_frame('NaN수').assign(비율=lambda x: (x['NaN수'] / len(df) * 100).round(2)))

print(f"\n[추가 컬럼 기술통계]")
df[ALL_NEW_COLS].describe().round(4)

최종 shape: (14823, 99)
추가된 컬럼 (21개): ['wpm', 'f0_mean', 'f0_std', 'hnr_mean', 'speech_ratio', 'face_area_ratio', 'face_count', 'avg_brightness', 'avg_saturation', 'contrast', 'text_area_ratio', 'color_variance', 'vtuber_features_score', 'vtuber_vibe_score', 'vtuber_styling_score', 'vtuber_technical_score', 'vtuber_rendering_score', 'vtuber_finish_score', 'vtuber_beauty_score', 'vtuber_qual_score', 'vtuber_appeal_score']

추가 피처 결측치 없음

[추가 컬럼 기술통계]


,wpm,f0_mean,f0_std,hnr_mean,speech_ratio,face_area_ratio,face_count,avg_brightness,avg_saturation,contrast,text_area_ratio,color_variance,vtuber_features_score,vtuber_vibe_score,vtuber_styling_score,vtuber_technical_score,vtuber_rendering_score,vtuber_finish_score,vtuber_beauty_score,vtuber_qual_score,vtuber_appeal_score
count,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000
mean,94.3195,282.8939,77.2521,13.3859,0.8983,0.0169,0.2532,0.4930,0.2568,0.3645,0.1192,4.1505,52.7425,48.7681,51.2126,50.3115,51.8960,50.8194,51.3576,51.4484,51.3576
std,30.8242,53.9658,11.6061,2.5564,0.0917,0.0486,0.5439,0.0989,0.0864,0.0421,0.0996,0.6317,15.8329,13.7447,13.5613,15.5970,15.4677,13.4691,13.9135,15.0510,13.9135
min,7.2078,137.1404,43.4347,6.5591,0.5716,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,7.2584,11.5081,18.5734,16.7401,14.2221,21.0304,6.7490,15.0679,6.7490
25%,65.9588,265.2978,73.2028,11.8913,0.8490,0.0000,0.0000,0.4338,0.1995,0.3440,0.0314,3.8340,47.5680,37.4263,41.5994,36.7382,41.0950,41.1910,41.6810,41.2791,41.6810
50%,93.8361,298.9263,79.7109,13.0514,0.9289,0.0000,0.0000,0.5020,0.2504,0.3711,0.1103,4.2039,55.1389,48.4038,50.4492,50.4363,52.0927,50.3928,54.2254,51.3003,54.2254
75%,119.5394,317.4003,85.0402,14.5509,0.9614,0.0000,0.0000,0.5618,0.3080,0.3927,0.1820,4.5524,63.1523,59.6293,60.0507,62.5696,62.2788,61.0271,61.3248,60.7958,61.3248
max,155.5129,373.8090,107.9751,23.2830,0.9981,0.6539,6.0000,0.8840,0.7621,0.4933,0.8000,6.1650,84.1219,79.8767,84.0463,85.5094,92.1512,81.4227,79.5172,93.1570,79.5172


---
## 5. 저장

In [ ]:
# 병합 결과 저장
OUT_PATH = BASE_PATH / "processed/video_processed.csv"
df.to_csv(OUT_PATH, sep="\x01", index=False, encoding='utf-8-sig')

# 저장 검증
df_check = pd.read_csv(OUT_PATH, sep="\x01", nrows=5, encoding='utf-8-sig')
assert df_check.shape[1] == df.shape[1], f"컬럼 수 불일치: {df_check.shape[1]} != {df.shape[1]}"

print(f"저장 완료: {OUT_PATH.name}")
print(f"shape: {df.shape}")
print(f"읽기 검증 컬럼 수: {df_check.shape[1]}")